# 29 — Image Captioning with an RNN

An image-captioning model conditions a sequence model on an image. This notebook uses small synthetic image feature vectors so we can focus on the captioning pipeline rather than a pretrained CNN or a large dataset.

## Learning objectives

- represent variable-length captions with special tokens and padding;
- align caption inputs, next-token targets, and a padding mask;
- project image features into an RNN's initial hidden state;
- connect embedding, recurrent, temporal-affine, and masked-loss layers;
- backpropagate through the complete captioning pipeline;
- generate a caption autoregressively with greedy decoding.


In [1]:
import numpy as np

from cs231n_practice.gradient_check import (
    eval_numerical_gradient_array,
    relative_error,
)
from cs231n_practice.rnn_layers import (
    rnn_backward,
    rnn_forward,
    rnn_step_forward,
)
from cs231n_practice.sequence_layers import (
    embedding_backward,
    embedding_forward,
    temporal_affine_backward,
    temporal_affine_forward,
    temporal_softmax_loss,
)

SEED = 42
generator = np.random.default_rng(SEED)


## 1. Caption tokens, shifting, and masking

Captioning uses three special tokens:

- `<START>` tells the model to begin generating;
- `<END>` tells it that the caption is complete;
- `<PAD>` fills unused array positions so captions can share one shape.

For a stored caption `[<START>, cat, sleeps, <END>, <PAD>]`, the model receives every token except the last and predicts every token except the first:

```text
input:   <START>  cat     sleeps  <END>
target:  cat      sleeps  <END>   <PAD>
mask:    True     True    True    False
```

The mask is based on the targets because it tells the loss which predictions correspond to real next tokens.

### Exercise 1 — Create caption inputs, targets, and mask


In [11]:
index_to_token = ["<PAD>", "<START>", "<END>", "cat", "dog", "runs", "sleeps"]
token_to_index = {token: index for index, token in enumerate(index_to_token)}
PAD = token_to_index["<PAD>"]
START = token_to_index["<START>"]
END = token_to_index["<END>"]

captions = np.array([
    [START, token_to_index["cat"], token_to_index["sleeps"], END, PAD],
    [START, token_to_index["dog"], token_to_index["runs"], END, PAD],
    [START, token_to_index["cat"], token_to_index["runs"], END, PAD],
])

# Inputs omit the final stored token; targets omit <START> and are shifted
# one position ahead. Only non-padding targets represent real predictions.
caption_inputs = captions[:, :-1]
caption_targets = captions[:, 1:]
caption_mask = caption_targets != PAD

assert caption_inputs.shape == caption_targets.shape == caption_mask.shape == (3, 4)
np.testing.assert_array_equal(caption_inputs, captions[:, :-1])
np.testing.assert_array_equal(caption_targets, captions[:, 1:])
np.testing.assert_array_equal(caption_mask, caption_targets != PAD)


## 2. The image initializes the recurrent state

Assume a CNN has already converted each image into a feature vector $F_n\in\mathbb{R}^{D_{image}}$. A learned affine projection converts that vector into the initial hidden state:

$$
h_0=F W_{image}+b_{image}.
$$

The shapes are

$$
(N,D_{image})(D_{image},H)+(H,)\longrightarrow(N,H).
$$

This is how information from the image enters the vanilla RNN. After initialization, the recurrent state carries that information forward while caption tokens enter through the embedding layer.

### Exercise 2 — Implement the image projection and its backward pass


In [15]:
def image_projection_forward(features, weights, bias):
    """Project image features into the initial hidden state."""
    # calculate h0 and cache what backward needs.
    output = features @ weights + bias
    cache = (features, weights)
    return output, cache


def image_projection_backward(dh0, cache):
    """Backpropagate from the initial hidden state to the projection."""
    # return dfeatures, dweights, and dbias.
    features, weights = cache
    dfeatures = dh0 @ weights.T
    dweights = features.T @ dh0
    dbias = np.sum(dh0, axis=0)
    return dfeatures, dweights, dbias


image_features = generator.normal(size=(3, 4))
image_weights = generator.normal(scale=0.2, size=(4, 5))
image_bias = generator.normal(scale=0.1, size=5)
h0, image_cache = image_projection_forward(image_features, image_weights, image_bias)
dh0 = generator.normal(size=h0.shape)
image_gradients = image_projection_backward(dh0, image_cache)

assert h0.shape == (3, 5)
for gradient, shape in zip(
    image_gradients,
    (image_features.shape, image_weights.shape, image_bias.shape),
):
    assert gradient.shape == shape

numerical_dfeatures = eval_numerical_gradient_array(
    lambda value: image_projection_forward(value, image_weights, image_bias)[0],
    image_features,
    dh0,
)
projection_error = relative_error(image_gradients[0], numerical_dfeatures)
print(f"image-projection gradient error: {projection_error:.3e}")
assert projection_error < 1e-8


image-projection gradient error: 3.171e-11


## 3. Complete captioning forward pass

The forward path is a composition of familiar layers:

```text
features ──> image projection ──> h0
                                     \
caption inputs ──> embeddings ──> RNN ──> temporal affine ──> scores ──> loss
```

With feature dimension $D_{image}$, embedding dimension $D$, hidden dimension $H$, and vocabulary size $V$, trace the shapes:

```text
features        (N, D_image)
caption inputs  (N, T)
embeddings      (N, T, D)
h0              (N, H)
hidden states   (N, T, H)
scores          (N, T, V)
loss            scalar
```

### Exercise 3 — Connect the reusable forward layers


In [19]:
def captioning_loss_forward(features, caption_inputs, caption_targets, mask, parameters):
    """Run an image-conditioned vanilla RNN and return its loss and cache."""
    # unpack parameters and call, in order:
    embedding_matrix = parameters["embedding_matrix"]
    image_weights = parameters["image_weights"]
    image_bias = parameters["image_bias"]
    rnn_weights_x = parameters["rnn_weights_x"]
    rnn_weights_h = parameters["rnn_weights_h"]
    rnn_bias = parameters["rnn_bias"]
    output_weights = parameters["output_weights"]
    output_bias = parameters["output_bias"]

    # 1. image_projection_forward
    h0, image_cache = image_projection_forward(features, image_weights, image_bias)

    # 2. embedding_forward
    input_vectors, embedding_cache = embedding_forward(caption_inputs, embedding_matrix)

    # 3. rnn_forward
    hidden_states, rnn_cache = rnn_forward(
        input_vectors, h0, rnn_weights_x, rnn_weights_h, rnn_bias,
    )
    
    # 4. temporal_affine_forward
    scores, temporal_cache = temporal_affine_forward(
        hidden_states, output_weights, output_bias
    )

    # 5. temporal_softmax_loss
    loss, dscores = temporal_softmax_loss(scores, caption_targets, mask)

    # Cache the image-projection, embedding, RNN, and temporal-affine
    # caches in that order for captioning_loss_backward.
    cache = (image_cache, embedding_cache, rnn_cache, temporal_cache)

    # Return (loss, dscores, cache).
    return loss, dscores, cache


## 4. Backpropagation through the complete model

Backward propagation follows the forward graph in reverse:

```text
dscores
   ↓ temporal affine backward
dhidden
   ↓ RNN backward
dembeddings and dh0
   ↓                 ↓
embedding backward  image projection backward
```

`rnn_backward` returns `dh0` because the initial hidden state is an input to the recurrent computation. In captioning, `h0` came from the image projection, so `dh0` continues backward into the image projection parameters and image features.

### Exercise 4 — Implement the complete backward pass


In [20]:
def captioning_loss_backward(dscores, cache):
    """Return gradients for every trainable captioning parameter."""
    # unpack the four forward caches and reverse the temporal-affine,
    image_cache, embedding_cache, rnn_cache, temporal_cache = cache

    # RNN, embedding, and image-projection operations. rnn_backward returns
    dhidden, doutput_weights, doutput_bias = temporal_affine_backward(dscores, temporal_cache)

    # (dembeddings, dh0, drnn_weights_x, drnn_weights_h, drnn_bias).
    embedding_upstream, dh0, drnn_weights_x, drnn_weights_h, drnn_bias = rnn_backward(dhidden, rnn_cache)

    _, dimage_weights, dimage_bias = image_projection_backward(dh0, image_cache)

    dembeddings = embedding_backward(embedding_upstream, embedding_cache)

    # Return a gradient dictionary with the same eight keys as parameters;
    gradients = {
        "embedding_matrix": dembeddings,
        "image_weights": dimage_weights,
        "image_bias": dimage_bias,
        "rnn_weights_x": drnn_weights_x,
        "rnn_weights_h": drnn_weights_h,
        "rnn_bias": drnn_bias,
        "output_weights": doutput_weights,
        "output_bias": doutput_bias,
    }

    return gradients


## 5. Small synthetic model check

The following parameters are random, so the loss will not yet be small. The purpose is to verify that the complete forward and backward pipeline has consistent shapes before training.


In [21]:
vocabulary_size = len(index_to_token)
embedding_dim = 4
hidden_dim = 5
parameters = {
    "embedding_matrix": generator.normal(scale=0.1, size=(vocabulary_size, embedding_dim)),
    "image_weights": generator.normal(scale=0.1, size=(image_features.shape[1], hidden_dim)),
    "image_bias": np.zeros(hidden_dim),
    "rnn_weights_x": generator.normal(scale=0.1, size=(embedding_dim, hidden_dim)),
    "rnn_weights_h": generator.normal(scale=0.1, size=(hidden_dim, hidden_dim)),
    "rnn_bias": np.zeros(hidden_dim),
    "output_weights": generator.normal(scale=0.1, size=(hidden_dim, vocabulary_size)),
    "output_bias": np.zeros(vocabulary_size),
}

caption_loss, dscores, caption_cache = captioning_loss_forward(
    image_features, caption_inputs, caption_targets, caption_mask, parameters
)
caption_gradients = captioning_loss_backward(dscores, caption_cache)

assert np.isfinite(caption_loss)
assert caption_gradients.keys() == parameters.keys()
for name in parameters:
    assert caption_gradients[name].shape == parameters[name].shape
print(f"untrained captioning loss: {caption_loss:.4f}")


untrained captioning loss: 5.8388


## 6. Autoregressive greedy sampling

Training processes all known caption inputs in parallel. At inference time, the target caption is unknown, so generation is sequential:

1. project the image into $h_0$;
2. begin with `<START>`;
3. embed the current token and take one RNN step;
4. project the new hidden state into vocabulary scores;
5. choose the highest-scoring token;
6. feed that token back as the next input;
7. stop at `<END>` or the maximum length.

This is greedy decoding: it makes the best immediate choice without reconsidering earlier choices.

### Exercise 5 — Implement greedy caption sampling


In [ ]:
# size 2 fill with START=1
np.full(2, START, dtype=int)

array([1, 1])

In [43]:
# reshape cols-rows
np.full(2, START, dtype=int)[:, None]

array([[1],
       [1]])

In [ ]:
# get embedding
embedding_forward(np.full(2, START, dtype=int)[:, None], parameters["embedding_matrix"])[0]

array([[[-0.22912895,  0.03043667,  0.00720336,  0.04138903]],

       [[-0.22912895,  0.03043667,  0.00720336,  0.04138903]]])

In [44]:
# 2 samples, 1 sequence step, 4-dim embedding size
embedding_forward(np.full(2, START, dtype=int)[:, None], parameters["embedding_matrix"])[0].shape

(2, 1, 4)

In [ ]:
# delete 1 sequence dimension
embedding_forward(np.full(2, START, dtype=int)[:, None], parameters["embedding_matrix"])[0][:, 0, :].shape

(2, 4)

In [47]:
def sample_caption_greedy(features, parameters, start_id, end_id, max_length):
    """Generate one caption per image using greedy next-token choices."""
    embedding_matrix = parameters["embedding_matrix"]
    image_weights = parameters["image_weights"]
    image_bias = parameters["image_bias"]
    rnn_weights_x = parameters["rnn_weights_x"]
    rnn_weights_h = parameters["rnn_weights_h"]
    rnn_bias = parameters["rnn_bias"]
    output_weights = parameters["output_weights"]
    output_bias = parameters["output_bias"]
    num_examples = features.shape[0]

    # The projected image features provide one initial hidden state per
    # image: (N, D_image) -> (N, H). Ignore the training cache here.
    h, _ = image_projection_forward(features, image_weights, image_bias)

    # Every caption begins with <START>. Prefill the output with <END> so
    # positions after an early stop remain harmless and the shape is fixed.
    current_ids = np.full(num_examples, start_id, dtype=int)
    sampled_ids = np.full((num_examples, max_length), end_id, dtype=int)
    finished = np.zeros(num_examples, dtype=bool)

    for t in range(max_length):
        # embedding_forward expects token IDs shaped (N, T). Add a one-step
        # time axis, then remove it because rnn_step_forward expects (N, D).
        embedded, _ = embedding_forward(current_ids[:, None], embedding_matrix)
        x_t = embedded[:, 0, :]

        h, _ = rnn_step_forward(
            x_t, h, rnn_weights_x, rnn_weights_h, rnn_bias
        )

        # One hidden state per image produces one vocabulary-score row.
        # Softmax is unnecessary: it preserves the ordering used by argmax.
        scores = h @ output_weights + output_bias
        next_ids = np.argmax(scores, axis=1)

        # Captions that ended earlier remain <END>; active captions accept
        # their newly predicted token.
        # Keep already completed captions at <END>.
        next_ids = np.where(finished, end_id, next_ids)

        # Save this step's token for every caption.
        sampled_ids[:, t] = next_ids

        # Mark captions that have now generated <END>.
        finished |= next_ids == end_id

        # Feed this step's predictions into the next generation step.
        current_ids = next_ids

        if np.all(finished):
            break

    return sampled_ids



sampled_ids = sample_caption_greedy(
    image_features[:2], parameters, START, END, max_length=6
)
assert sampled_ids.shape == (2, 6)
assert np.issubdtype(sampled_ids.dtype, np.integer)
for row in sampled_ids:
    words = []
    for token_id in row:
        if token_id == END:
            break
        words.append(index_to_token[token_id])
    print(" ".join(words))


runs <START> runs dog <PAD> cat
dog <PAD> cat <START> runs dog


## 7. Reflection

1. Why is an image projected into $h_0$ instead of treated as an ordinary caption token here?
2. Why do caption inputs begin with `<START>` while targets do not?
3. Why is the mask constructed from caption targets rather than caption inputs?
4. Through which gradient does the caption loss reach the image projection?
5. Why are the embedding and output-projection matrices different even though both involve the vocabulary?
6. Why can training process all caption time steps together while sampling proceeds one step at a time?
7. What is the main limitation of greedy decoding?
8. Why should random untrained parameters produce meaningless captions?

### Answers

1. The image feature vector describes the complete image rather than one word in the caption. Projecting it into $h_0$ gives the RNN image context before it processes the first caption token. Other captioning architectures can represent images as tokens, but that is not the design used here.
2. `<START>` provides the first input when no previous word exists. The corresponding first target is the first real caption word, so `<START>` is not itself something the model should predict.
3. The mask controls which next-token predictions contribute to the loss, and those predictions are compared with `caption_targets`. A target equal to `<PAD>` is not a real prediction task and must contribute neither loss nor gradient.
4. The caption loss first produces `dscores`, which flows backward through the temporal projection and the RNN. `rnn_backward` returns $dh_0$, and this gradient then enters `image_projection_backward` to produce gradients for the image-projection weights, bias, and input features.
5. The embedding matrix maps a token ID to an input vector of dimension $D$, whereas the output matrix maps a hidden vector of dimension $H$ to one score per vocabulary token. They perform opposite roles and can have different dimensions, although some language models deliberately tie compatible input and output weights.
6. During training, teacher forcing provides the complete sequence of correct input tokens in advance, so their embeddings and output losses can be represented in batched tensors. During sampling, the next input is the model's previous prediction, so it is unknown until the preceding step has been computed.
7. Greedy decoding selects the best token at the current step without considering whether a lower-scoring choice could lead to a better complete caption. It can therefore miss globally better sequences.
8. Random parameters have not learned relationships between image features, caption prefixes, and likely next tokens. Their vocabulary scores are effectively arbitrary, so generated captions may contain unrelated words and even special tokens such as `<START>` or `<PAD>`.
